In [ ]:
import numpy as np
import scipy.stats as stats
from scipy.special import logsumexp
import pickle
from auditing_utils import get_privacy_profile

In [ ]:
## Hidden-State Simulation (vectorized)

def ll_pq_substitute(g_T, T, C, sigma, q, sign=+1):
    """
    log p(g_T | dataset) for the substitute relation.
    sign = +1  -> D  (poisoned steps shift by +C)
    sign = -1  -> D' (poisoned steps shift by -C)
    """
    x = np.asarray(g_T)
    std = sigma * abs(C) * np.sqrt(T)
    ks = np.arange(T + 1)                                
    logw = stats.binom.logpmf(ks, T, q)                  
    means = sign * ks * C                                
    logpdf = stats.norm.logpdf(x[..., None], loc=means[None, :], scale=std)
    return logsumexp(logw + logpdf, axis=-1)

def final_step_llr(g_T, T, C, sigma, q):
    """
    LLR = log p(g_T | D) - log p(g_T | D') under the substitute relation.
    Accepts scalar or array O_T.
    """
    ld  = ll_pq_substitute(g_T, T, C, sigma, q, sign=+1)
    ldp = ll_pq_substitute(g_T, T, C, sigma, q, sign=-1)
    return ld - ldp

def get_likelihoods(R, T, sigma, q, C, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    observations_D = np.zeros((R, T))
    observations_D_prime = np.zeros((R, T))
    for r in range(R):
        poisoned = rng.binomial(1, q, size=T).astype(bool)   
        Z = rng.normal(0.0, sigma * abs(C), size=T)          

        ## poison following SUBSTITUTE adjacency:
        X_D  = Z + np.where(poisoned, +C, 0.0)               
        X_Dp = Z + np.where(poisoned, -C, 0.0) 

        observations_D[r]        = np.cumsum(X_D)
        observations_D_prime[r]  = np.cumsum(X_Dp)
    llr_D_final  = final_step_llr(observations_D[:, -1],  T, C, sigma, q)    # log P(·|D) - log P(·|D')
    llr_Dprime_final = final_step_llr(observations_D_prime[:, -1], T, C, sigma, q)
    return llr_D_final, llr_Dprime_final

def get_privacy_profile_per_step(scores,gts,delta:float=1e-5):
    _, _, _, empirical_epsilons_gdp = get_privacy_profile(
                                                delta=delta, 
                                                outputs=scores, 
                                                gt=gts,
                                                alpha=0.05, 
                                                method="beta", 
                                            )

    return np.max(empirical_epsilons_gdp)

In [ ]:
def find_smallest_epsilon_from_grid(
    deltas_grid,
    epsilons_grid,
    delta_target: float,
    tol_rel: float = 1e-12,
    tol_abs: float = 0.0) :
    if not (delta_target > 0.0 and delta_target <= 1):
        raise ValueError("delta_target must be a between [0,1].")

    ## compute RHS = (1 + exp(eps)) * delta_target, guarding overflow
    rhs = np.empty_like(epsilons_grid)
    with np.errstate(over="ignore"):
        exp_e = np.exp(epsilons_grid)
    
    rhs = (1.0 + exp_e) * deltas_grid  # inf if exp overflowed

    ## map tolerance per point
    tol = np.maximum(tol_abs, tol_rel * np.abs(rhs))

    ## probable delta with diff > 0 and within tolerance matches
    diff = delta_target - rhs
    delta_matches = np.isfinite(diff) & (np.abs(diff) <= -tol)
 
    if np.any(delta_matches):
        ## choose the smallest epsilon among matches (array already sorted by epsilon)
        idx = int(np.nonzero(delta_matches)[0][0])
        status = "ok"
    else:
        ## choose the closest grid point by absolute gap and ignore non-finite diffs caused by overflow
        finite = np.isfinite(diff)
        if not np.any(finite):
            raise ValueError("All RHS values overflowed; epsilons are too large for exp().")
        idx = int(np.nanargmin(np.abs(diff[finite])))
        ## map back to full index among finite subset
        idx = int(np.nonzero(finite)[0][idx])
        status = "approx"

    return {
        "epsilon_ar_prime": epsilons_grid[idx],
        "delta_ar_prime": deltas_grid[idx],
        "status": status,
        "gap": diff[idx],
    }

In [ ]:
from gdpnum.subst_prv import PoissonSubsampledGaussianMechanismSubstitute
from prv_accountant import PRVAccountant, PoissonSubsampledGaussianMechanism

# ar_epsilons = [0.125,0.25,0.5,1,2,4]
seeds = 3
delta_target = 1e-5
q = 1.0
T = 500
R = 25000
C = 1.0

sigmas = {1.0:[168.076171875, 89.12841796875, 48.4033203125, 34.129638671875, 26.89208984375],
         0.25:[42.080078125, 22.34832763671875, 12.1148681640625, 8.54278564453125, 6.693878173828125],
         0.0625:[10.4949951171875, 5.5780029296875, 3.017425537109375, 2.1319961547851562, 1.6772842407226562]}

N = len(sigmas[1.0])
ar_epsilons, res_audit_subst_eps, res_accounting_subst_eps = np.zeros(N), np.zeros((N,seeds)), np.zeros(N)

for i in range(N):
    for s in range(seeds):
        sigma = sigmas[q][i]
        ## get audit epsilon for the final step
        likelihoods_D, likelihoods_D_prime = get_likelihoods(R,T,sigma,q,C)
        scores,gts = np.concatenate([likelihoods_D,likelihoods_D_prime]), np.concatenate([np.zeros(R),np.ones(R)])
        audit_subst_epsilon = get_privacy_profile_per_step(scores,gts,delta_target)
        res_audit_subst_eps[i][s] = audit_subst_epsilon
        if s == 0:
            ## compute accounting substitute epsilon equivalent
            subst_prv = PoissonSubsampledGaussianMechanismSubstitute(sampling_probability=q, noise_multiplier=sigma)
            accountant = PRVAccountant(prvs=subst_prv, max_self_compositions=T, eps_error=1e-4, delta_error=1e-10)
            accounting_subst_epsilon = accountant.compute_epsilon(delta=delta_target, num_self_compositions=T)[-1] 
            res_accounting_subst_eps[i] = accounting_subst_epsilon  

            ## compute accounting add/remove epsilon equivalent
            ar_prv = PoissonSubsampledGaussianMechanism(sampling_probability=q, noise_multiplier=sigma)
            accountant = PRVAccountant(prvs=ar_prv, max_self_compositions=T, eps_error=1e-4, delta_error=1e-10)
            accounting_ar_epsilon = accountant.compute_epsilon(delta=delta_target, num_self_compositions=T)[-1] 
            ar_epsilons[i] = accounting_ar_epsilon  

accounting_subst_by_conversion = np.zeros(N)
for i in range(N):
    sigma = sigmas[q][i]
    ## epsilon expected for substitute adjacency with delta adjustment
    deltas = np.geomspace(1e-9,1e-4,num=100)
    epsilons = np.zeros(len(deltas))
    for d in range(deltas.shape[0]):
        ar_prv = PoissonSubsampledGaussianMechanism(sampling_probability=q, noise_multiplier=sigma)
        accountant = PRVAccountant(prvs=ar_prv, max_self_compositions=T, eps_error=1e-3, delta_error=1e-10)
        epsilons[d] = accountant.compute_epsilon(delta=deltas[d], num_self_compositions=T)[-1]
    res = find_smallest_epsilon_from_grid(deltas,epsilons,delta_target,tol_rel=1e-10)
    accounting_subst_by_conversion[i] = 2. * res["epsilon_ar_prime"]   

results = {
            "ar_epsilons" : ar_epsilons,
            "accounting_subst_by_conversion":accounting_subst_by_conversion,
            "accounting_subst_epsilon": res_accounting_subst_eps,
            "auditing_subst_epsilon":res_audit_subst_eps
           }

with open(f"../results/accounting_results_@_q_{q}_T_{T}_R_{R}.pkl", "wb") as f:
    pickle.dump(results,f)  